# Robustness check — target encoding leakage (subsampled RF) #

Refits the subsampled RF on every layer **without** `origin_te` / `dest_te` and compares the
validation-to-test gap (2024 OOB AUC-ROC → 2025 AUC-ROC) with the original runs.

**Logic:** if the out-of-fold target encoding leaked label information into 2024 validation
predictions, OOB AUC would be inflated *only* when the encodings are present, so the gap
with encodings would be clearly larger than the gap without them. Absolute AUC is expected
to drop without the encodings (airport information is lost), so compare **gaps, not levels**.

Baseline ("with TE") numbers are read from the saved `*_probs.npz` of `classical_ML_subsample.ipynb`
(no refit). Note: `oob_score_` in those runs is OOB *accuracy*; here OOB AUC is computed from the
saved OOB probabilities.

In [1]:
from pathlib import Path
import time
import json
import gc
import sys

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             matthews_corrcoef, precision_recall_curve)

scripts_dir = (Path.cwd() / ".." / "scripts").resolve()
sys.path.append(str(scripts_dir))
from config import final_data_path

final_data_path = Path(final_data_path)
features_dir = final_data_path
baseline_dir = final_data_path / "modeling_results" / "rf"          # original subsampled RF outputs
results_dir = final_data_path / "modeling_results" / "rf_robustness_no_te"
results_dir.mkdir(parents=True, exist_ok=True)

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]
te_cols = ["origin_te", "dest_te"]

subsampled_layers = [
    "layer_a_bts_subsampled",
    "layer_b_bts_meteostat_subsampled",
    "layer_c_bts_gdelt_subsampled",
    "layer_d_bts_meteostat_gdelt_subsampled",
]

# Same config as the main subsampled RF runs
rf_params = {
    "n_estimators": 300,
    "max_depth": 30,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "oob_score": True,
    "n_jobs": 4,
    "random_state": 42,
}
beta = 1.2


def get_train_test(key: str, drop_cols=()):
    """Load one layer, split by year, drop non-feature (and optionally extra) columns."""
    df = pd.read_parquet(features_dir / f"{key}_features.parquet")
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]
    del df
    feature_cols = [c for c in train.columns if c not in non_feature_cols and c not in drop_cols]
    return train[feature_cols], train[target_col], test[feature_cols], test[target_col]


def oob_metrics(y, proba):
    """AUC-ROC / AUC-PR on OOB probabilities, ignoring rows never out-of-bag (NaN)."""
    y, proba = np.asarray(y), np.asarray(proba)
    mask = np.isfinite(proba)
    return roc_auc_score(y[mask], proba[mask]), average_precision_score(y[mask], proba[mask])


def tune_threshold(y_true, oob_proba, beta=beta):
    """F-beta-optimal threshold on OOB predictions (same rule as the main runs)."""
    y_true, oob_proba = np.asarray(y_true), np.asarray(oob_proba)
    mask = np.isfinite(oob_proba)
    precision, recall, thresholds = precision_recall_curve(y_true[mask], oob_proba[mask])
    fbeta = (1 + beta**2) * precision * recall / (beta**2 * precision + recall + 1e-12)
    return thresholds[np.nanargmax(fbeta[:-1])]


def summarise(y_train, oob_proba, y_test, test_proba):
    oob_auc, oob_ap = oob_metrics(y_train, oob_proba)
    threshold = tune_threshold(y_train, oob_proba)
    test_pred = (np.asarray(test_proba) >= threshold).astype(int)
    test_auc = roc_auc_score(y_test, test_proba)
    return {
        "oob_auc_roc": oob_auc,
        "test_auc_roc": test_auc,
        "gap_auc_roc": oob_auc - test_auc,
        "oob_auc_pr": oob_ap,
        "test_auc_pr": average_precision_score(y_test, test_proba),
        "test_f1": f1_score(y_test, test_pred),
        "test_mcc": matthews_corrcoef(y_test, test_pred),
        "threshold": float(threshold),
    }

/opt/anaconda3/envs/thesis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Baseline — **with** target encodings, from the saved probabilities of the main runs.

In [2]:
with_te = {}
for key in subsampled_layers:
    saved = np.load(baseline_dir / f"{key}_probs.npz")
    with_te[key] = summarise(saved["y_train"], saved["oob_proba"], saved["y_test"], saved["test_proba"])
    print(f"{key}: OOB AUC {with_te[key]['oob_auc_roc']:.4f} -> "
          f"2025 AUC {with_te[key]['test_auc_roc']:.4f} (gap {with_te[key]['gap_auc_roc']:.4f})")

layer_a_bts_subsampled: OOB AUC 0.7080 -> 2025 AUC 0.6731 (gap 0.0348)
layer_b_bts_meteostat_subsampled: OOB AUC 0.7232 -> 2025 AUC 0.6856 (gap 0.0376)
layer_c_bts_gdelt_subsampled: OOB AUC 0.7257 -> 2025 AUC 0.6772 (gap 0.0485)
layer_d_bts_meteostat_gdelt_subsampled: OOB AUC 0.7296 -> 2025 AUC 0.6860 (gap 0.0436)


Refit **without** `origin_te` / `dest_te` (≈ 3 min per layer, ~15 min total).

In [3]:
without_te = {}
for key in tqdm(subsampled_layers, desc="RF without TE"):
    X_train, y_train, X_test, y_test = get_train_test(key, drop_cols=te_cols)
    assert not set(te_cols) & set(X_train.columns)
    print(f"\n{key}: train {X_train.shape}, test {X_test.shape}")

    model = RandomForestClassifier(**rf_params)
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start

    oob_proba = model.oob_decision_function_[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]

    without_te[key] = {**summarise(y_train, oob_proba, y_test, test_proba), "fit_time_s": elapsed}
    print(f"OOB AUC {without_te[key]['oob_auc_roc']:.4f} -> 2025 AUC "
          f"{without_te[key]['test_auc_roc']:.4f} (gap {without_te[key]['gap_auc_roc']:.4f}) "
          f"| fit {elapsed:.0f}s")

    np.savez(results_dir / f"{key}_no_te_probs.npz",
             y_train=y_train.to_numpy(), oob_proba=oob_proba,
             y_test=y_test.to_numpy(), test_proba=test_proba)
    with open(results_dir / f"{key}_no_te_metrics.json", "w") as f:
        json.dump(without_te[key], f, indent=2)

    del model, X_train, X_test, y_train, y_test, oob_proba, test_proba
    gc.collect()

RF without TE:   0%|          | 0/4 [00:00<?, ?it/s]


layer_a_bts_subsampled: train (1000000, 23), test (5966559, 23)
OOB AUC 0.6937 -> 2025 AUC 0.6597 (gap 0.0340) | fit 142s


RF without TE:  25%|██▌       | 1/4 [03:01<09:04, 181.59s/it]


layer_b_bts_meteostat_subsampled: train (1000000, 27), test (5966559, 27)
OOB AUC 0.7181 -> 2025 AUC 0.6792 (gap 0.0389) | fit 183s


RF without TE:  50%|█████     | 2/4 [06:57<07:07, 213.77s/it]


layer_c_bts_gdelt_subsampled: train (1000000, 38), test (5966559, 38)
OOB AUC 0.7219 -> 2025 AUC 0.6710 (gap 0.0509) | fit 213s


RF without TE:  75%|███████▌  | 3/4 [11:24<03:57, 237.93s/it]


layer_d_bts_meteostat_gdelt_subsampled: train (1000000, 42), test (5966559, 42)
OOB AUC 0.7253 -> 2025 AUC 0.6820 (gap 0.0432) | fit 232s


RF without TE: 100%|██████████| 4/4 [16:13<00:00, 243.41s/it]


Comparison table.

In [4]:
rows = []
for key in subsampled_layers:
    w, wo = with_te[key], without_te[key]
    rows.append({
        "layer": key.replace("_subsampled", ""),
        "oob_auc_with_te": w["oob_auc_roc"],
        "test_auc_with_te": w["test_auc_roc"],
        "gap_with_te": w["gap_auc_roc"],
        "oob_auc_no_te": wo["oob_auc_roc"],
        "test_auc_no_te": wo["test_auc_roc"],
        "gap_no_te": wo["gap_auc_roc"],
        "gap_difference": w["gap_auc_roc"] - wo["gap_auc_roc"],   # > 0 would point to TE-specific inflation
        "test_auc_contribution_of_te": w["test_auc_roc"] - wo["test_auc_roc"],
    })

comparison = pd.DataFrame(rows).set_index("layer")
comparison.to_csv(results_dir / "te_robustness_comparison.csv")
comparison.round(4)

,oob_auc_with_te,test_auc_with_te,gap_with_te,oob_auc_no_te,test_auc_no_te,gap_no_te,gap_difference,test_auc_contribution_of_te
layer,,,,,,,,
layer_a_bts,0.7080,0.6731,0.0348,0.6937,0.6597,0.0340,0.0008,0.0135
layer_b_bts_meteostat,0.7232,0.6856,0.0376,0.7181,0.6792,0.0389,-0.0013,0.0064
layer_c_bts_gdelt,0.7257,0.6772,0.0485,0.7219,0.6710,0.0509,-0.0024,0.0062
layer_d_bts_meteostat_gdelt,0.7296,0.6860,0.0436,0.7253,0.6820,0.0432,0.0004,0.0040


**Reading the result**

- `gap_difference` ≈ 0 (well below the 0.005 minimum meaningful AUC difference used elsewhere):
  the OOB→2025 drop is the same with and without encodings → it reflects 2024→2025 drift,
  not leakage through the target encoding.
- `gap_difference` clearly positive: OOB validation was inflated by the encodings → the
  threshold/hyperparameter selection should be revisited.
- `test_auc_contribution_of_te` shows how much genuine out-of-sample signal the airport
  encodings carry on 2025, which cannot be inflated by leakage because 2025 labels never
  enter any encoding.
- The layer ranking (A < C < B ≈ D) should hold with and without encodings; if it does,
  the layer conclusions do not depend on the encoding.